# LUMIN Agents (OpenAI + OLMo)

This notebook wires the LUMIN query compiler to two LLM backends:

- OpenAI (via `OPENAI_API_KEY`)
- OLMo 7B Instruct (OpenAI-compatible endpoint via `OLMO_BASE_URL` and `OLMO_API_KEY`)

In [20]:
# Clone the repo into Colab
!git clone https://github.com/anhkos/LUMIN-v2-Search-Agent.git

fatal: destination path 'LUMIN-v2-Search-Agent' already exists and is not an empty directory.


In [21]:
%pip install -r "/content/LUMIN-v2-Search-Agent/requirements.txt"

In [22]:
import os
import sys

repo_root = "/content/LUMIN-v2-Search-Agent"
sys.path.insert(0, os.path.join(repo_root, "src"))
print("repo_root:", repo_root)

repo_root: /content/LUMIN-v2-Search-Agent


In [24]:
import json
import os
import sys
import ast
from dotenv import load_dotenv
from openai import OpenAI  # type: ignore

repo_root = "/content/LUMIN-v2-Search-Agent"
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from logic_engine import NeuroSymbolicSolver

load_dotenv()
ontology_path = os.path.join(repo_root, "data", "ontology.json")
solver = NeuroSymbolicSolver(ontology_path=ontology_path)

In [25]:
SYSTEM_PROMPT = """
You are the LUMIN Query Compiler for NASA PDS.
Your goal is to translate Natural Language into a Logic S-Expression.

### ONTOLOGY (Valid Concepts):
{ontology_keys}

### OPERATIONS:
1. INTERSECT(A, B) -> Returns overlap.
2. UNION(A, B) -> Returns combination.
3. DIFFERENCE(Base, Subtract) -> Removes constraints.

### RULES:
- Output ONLY the tuple plan. No markdown, no explanation.
- Use Concept Names EXACTLY as listed in the Ontology.
- If the user asks for a specific time/value not in ontology, ignore it for this MVP (or map to closest concept).

### EXAMPLES:
User: "Southern summer images"
Output: "Southern Summer"

User: "Southern summer but not polar regions"
Output: ('DIFFERENCE', 'Southern Summer', 'Polar Regions')

User: "Midnight observations during the dust storm season"
Output: ('INTERSECT', 'Midnight', 'Dust Storm Season')
"""

In [40]:
def _build_prompt():
    ontology_keys = ', '.join(solver.ontology.keys())
    return SYSTEM_PROMPT.format(ontology_keys=ontology_keys)

def _parse_plan(raw_plan):
    # Safer than eval for tuple parsing
    return ast.literal_eval(raw_plan)

def _field_for_term(term):
    if isinstance(term, str):
        return solver.ontology.get(term, {}).get('field')
    return None

def _normalize_plan(plan):
    if isinstance(plan, str):
        return plan
    op, arg1, arg2 = plan
    arg1 = _normalize_plan(arg1)
    arg2 = _normalize_plan(arg2)
    if op == 'INTERSECT':
        field1 = _field_for_term(arg1)
        field2 = _field_for_term(arg2)
        if field1 and field2 and field1 != field2:
            return ('UNION', arg1, arg2)
    return (op, arg1, arg2)

def run_agent(user_query, client, model):
    prompt = _build_prompt()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': user_query}
        ],
        temperature=0.0,
    )
    raw_plan = response.choices[0].message.content.strip()
    parsed_plan = _parse_plan(raw_plan)
    normalized_plan = _normalize_plan(parsed_plan)
    result = solver.execute_plan(normalized_plan)
    return raw_plan, result

In [32]:
def openai_agent(user_query, model='gpt-4o'):
    client = OpenAI()
    return run_agent(user_query, client, model)

def olmo_agent(user_query, model='allenai/olmo-3-7b-instruct'):
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        raise ValueError('Missing OPENROUTER_API_KEY env var')
    
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
        default_headers={
            'HTTP-Referer': os.getenv('OPENROUTER_SITE_URL', ''),
            'X-Title': os.getenv('OPENROUTER_SITE_NAME', '')
        }
    )
    
    return run_agent(user_query, client, model)

## Example Usage

In [35]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")

In [41]:
query = 'I need thermal data taken at midnight'

# OpenAI
openai_plan, openai_result = openai_agent(query)
print('OpenAI Plan:', openai_plan)
print('OpenAI Result:', json.dumps(openai_result, indent=2))

# OLMo (requires OLMO_BASE_URL to be set)
olmo_plan, olmo_result = olmo_agent(query)
print('OLMo Plan:', olmo_plan)
print('OLMo Result:', json.dumps(olmo_result, indent=2))

OpenAI Plan: "Midnight"
OpenAI Result: {
  "type": "cyclic_range",
  "field": "local_true_solar_time",
  "min": 23.0,
  "max": 1.0,
  "range_max": 24.0,
  "description": "Requires cyclic logic handling across the 24h boundary"
}
OLMo Plan: ('INTERSECT', 'Midnight', 'Raw Telemetry')
OLMo Result: [
  {
    "type": "cyclic_range",
    "field": "local_true_solar_time",
    "min": 23.0,
    "max": 1.0,
    "range_max": 24.0,
    "description": "Requires cyclic logic handling across the 24h boundary"
  },
  {
    "type": "categorical",
    "field": "processing_level",
    "values": [
      "Raw",
      "Telemetry"
    ],
    "description": "Uncalibrated instrument data"
  }
]
